# One GRPO Step on One SWE-bench Task

**Goal of Phase 1 (this notebook): conceptual clarity, not full scale SWE training.**  
**Goal of Phase 2 (next session): full scale SWE training.** 

By the end you will have watched a single gradient update travel the whole pipeline, with
every intermediate value printed.

| § | what it covers |
|---|---|
| 1 | One SWE-bench task, field by field |
| 2 | Environment |
| 3 | SWE Agent — `model + harness` |
| 4 | Reward |
| 5 | Group of rollouts |
| 6 | One GRPO update to the model |

Runs on a T4 in a few minutes; on CPU if you're patient.

In [ ]:
!pip -q install "transformers>=4.44" "datasets>=2.20" "accelerate>=0.33" "peft>=0.12" torch
# Colab ships torchao 0.10, below peft's floor -- peft raises on it. We don't use it.
!pip -q uninstall -y torchao

import os, urllib.request
if not os.path.exists("utils.py"):      # Colab: fetch the plumbing
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/abgoswam/swe_in_prod_vizuara_01/main/utils.py",
        "utils.py")

import json, random
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from utils import (TARGET, MOCK_FILE, MockEnv, SYSTEM, NO_COMMAND,
                   first_bash_block, generate, scripted_sampler,
                   show_rollout, show_group)

for _opt in ["display.max_colwidth", "display.max_rows", "display.max_columns", "display.width"]:
    pd.set_option(_opt, None)

random.seed(0); torch.manual_seed(0)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV)

---
# 1. Load one instance

We take one and read it field by field

In [ ]:
from datasets import load_dataset

ds = load_dataset("princeton-nlp/SWE-bench_Verified", split="test")
print(ds)

# A small, single-file task keeps the walkthrough readable.
cands = [i for i, r in enumerate(ds)
         if r["patch"].count("diff --git") == 1 and len(r["patch"]) < 1800]
inst = ds[cands[0]]
fail_to_pass = json.loads(inst["FAIL_TO_PASS"])
pass_to_pass = json.loads(inst["PASS_TO_PASS"])

display(pd.DataFrame(
    [(k, inst.get(k)) for k in ["instance_id", "repo", "base_commit", "version", "difficulty"]],
    columns=["field", "value"],
))

display(pd.DataFrame([
    ("1.1", "problem_statement", "the agent",  "the input: a raw GitHub issue"),
    ("-",   "patch",             "nobody",     "reference solution; unused in this notebook"),
    ("1.2", "test_patch",        "the grader", "adds the tests that define 'fixed'"),
    ("1.3", "FAIL_TO_PASS",      "the grader", "must go red -> green"),
    ("1.4", "PASS_TO_PASS",      "the grader", "must stay green"),
], columns=["section", "field", "who sees it", "job"]))

## 1.1 `problem_statement` — the agent's entire input

A raw GitHub issue. The agent is never told which file to open, or that
`separable.py` exists. Finding it — localization — is most of the real
difficulty of SWE-bench.

In [ ]:
print(inst["problem_statement"])

## 1.2 `test_patch` — the grader's tests (held out, always applied)

The real PR changed two things: `separable.py` (the fix) and `test_separable.py`
(the regression tests). 

SWE-bench splits that PR 
— the source half becomes `inst["patch"]`, which nothing here uses, 
— the test half becomes `test_patch`
— then pins `base_commit` to just before it.

So **these tests do not exist in the repository the agent works in.** Grading
has to inject them or nothing can detect the bug. The agent must not see them
either.

In [ ]:
print(inst["test_patch"])

## 1.3 `FAIL_TO_PASS` — must go red → green

The half of the reward that says *you solved the issue*.

In [ ]:
for t in fail_to_pass:
    print(t)

## 1.4 `PASS_TO_PASS` — must stay green

In [ ]:
for t in pass_to_pass:
    print(t)

In [ ]:
display(pd.DataFrame([
    ("existing cases",       6, "PASS_TO_PASS"),
    ("new, already green",   2, "PASS_TO_PASS"),
    ("new, red until fixed", 2, "FAIL_TO_PASS"),
    ("other tests in the file", 5, "PASS_TO_PASS"),
], columns=["group", "n_tests", "list"]))

print(f"\nFAIL_TO_PASS {len(fail_to_pass)}   PASS_TO_PASS {len(pass_to_pass)}")

---
# The whole picture

![SWE agent and environment interface](https://raw.githubusercontent.com/abgoswam/swe_in_prod_vizuara_01/main/swe_agent_env_interface.png)

Two loops. The **inner loop** is one episode: the agent sends an action, the
environment returns an observation, repeat. The **outer loop** takes the
finished trajectory plus its reward, turns a group of them into advantages, and
updates the policy.

Everything from here on builds one piece of this diagram — at smaller scale.
The diagram shows the real thing: a Docker container, a `submit` exit, a
50-turn budget, 16 rollouts per task. We have a Python dict, a fixed 4 turns,
and 6 rollouts.

---
# 2. The environment

$$\text{state} \;\xrightarrow{\ \text{action}\ }\; \text{state}' \;+\; \text{observation}$$

- **state** — the files. Here, exactly one: `separable.py`.
- **action** — a bash command.
- **observation** — what the command printed. All the agent ever sees.

| § | |
|---|---|
| 2.1 | the initial state |
| 2.2 | take an action |
| 2.3 | the new state |

In production the environment is a container with the repo cloned at
`base_commit`. Ours is a mock Python dict

## 2.1 Initial state

One file, 29 lines — the real `_cstack` from `separable.py` at `base_commit`.
The bug is the asymmetry: `cleft` is assigned `left`, but `cright` is assigned
`1`.

In [ ]:
env = MockEnv(fail_to_pass)

print("state = files in the environment:", list(env.fs), "\n")
print(env.fs[TARGET])

## 2.2 Take an action

An action is one bash command. The first three are read-only — they return an
observation and leave the state alone. Only the last one, a write, moves it.

In [ ]:
FIXED = MOCK_FILE.replace("= 1", "= right")     # the one-token fix

rows = []
for action in ["ls",
               f'grep -n "cright" {TARGET}',
               "python -m pytest",
               f"cat > {TARGET} <<'EOF'\n{FIXED}\nEOF"]:
    before = dict(env.fs)
    obs = env.run(action)
    rows.append({"action":        action.splitlines()[0] + (" ..." if "\n" in action else ""),
                 "state changed": env.fs != before,
                 "observation":   obs.replace("\n", " | ")[:60]})

display(pd.DataFrame(rows))

## 2.3 New state

The write landed. Here is the state now — and its difference from the initial
state **is** the candidate patch. Nothing else builds it.

In [ ]:
print("new state:\n")
print(env.fs[TARGET])

print("\ninitial state vs new state — the candidate patch:\n")
print(env.patch())

---
# 3. The agent

$$\textbf{agent} = \textbf{model} + \textbf{harness}$$

- **model** — text in, text out. Stateless. The only part that can learn.
- **harness** — deterministic code: prepare, parse, execute, append. Never trained.
- **rollout** — It is what you **get** when you run the agent once: one trajectory. `run_agent()` returns a rollout.

| § | what we build |
|---|---|
| 3.1 | **the model** — Qwen2.5-Coder-0.5B, the policy |
| 3.2 | **the harness** — the loop, and the system prompt that fixes the action space |
| 3.3 | **one rollout** — the two together, run once |

## 3.1 The model

Qwen2.5-Coder-0.5B-Instruct — the policy. 

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16 if DEV == "cuda" else torch.float32,
    attn_implementation="sdpa").to(DEV)

print(f"{MODEL}\n{sum(p.numel() for p in model.parameters()):,} parameters, none trainable yet")

## 3.2 The harness

=============

```python
context = [SYSTEM, ISSUE]

for turn in range(MAX_TURNS):            # our ONLY stopping condition
    prompt = chat_template(context)      # 1. prepare
    reply  = model.generate(prompt)      # 2. call   
    action = first_bash_block(reply)     # 3a. parse 
    if action is None:
        break
    obs    = env.run(action)             # 3b. execute
    context += [reply, obs]              # 4. append 

return context                           # one rollout
```

=============

The same loop in real code.

In [ ]:
# run_agent() drives the model through the harness. One call returns one
# rollout: the whole context, since in RL the trajectory is the training example.

def run_agent(model, tok, max_turns=4, temperature=1.0, sample=generate):
    env = MockEnv(fail_to_pass)
    context = [{"role": "system", "content": SYSTEM},
               {"role": "user",   "content": f"ISSUE:\n{inst['problem_statement'][:1500]}"}]

    for _ in range(max_turns):
        prompt = tok.apply_chat_template(context, tokenize=False, add_generation_prompt=True)
        reply  = sample(model, tok, prompt, temperature=temperature)
        action = first_bash_block(reply)
        obs    = env.run(action) if action else NO_COMMAND
        context += [{"role": "assistant", "content": reply},
                    {"role": "user",      "content": obs[:800]}]

    return dict(messages=context, patch=env.patch(), final=dict(env.fs), calls=env.calls)

## 3.3  What does a SWE rollout look like ?
Model + harness, run once against the environment. 

### 3.3.1 With the model we loaded

Read the **written by** column: three parties author one trajectory, and only
the `model` rows are the policy's own output.

A 0.5B model almost never reaches a write, so the state never moves and the
patch comes out empty.

In [ ]:
rollout = run_agent(model, tok)

In [ ]:
show_rollout(rollout)

### 3.3.2 With an oracle model

In [ ]:
SCRIPT = [
    f"Let me read the file.\n```bash\ncat {TARGET}\n```",
    f"The cright branch assigns 1 instead of right. Fixing it.\n"
    f"```bash\ncat > {TARGET} <<'EOF'\n{FIXED}\nEOF\n```",
    "Now run the tests.\n```bash\npython -m pytest\n```",
    "Check the tree.\n```bash\nls\n```",
]

expert = run_agent(model, tok, sample=scripted_sampler(SCRIPT))

In [ ]:
show_rollout(expert)

---
# 4. Reward

**The real reward** clones the repo at `base_commit`, applies `test_patch`,
applies the candidate patch, runs `FAIL_TO_PASS` and `PASS_TO_PASS`, and returns
`1.0` only if every test in both lists passes. Binary, no partial credit.

We cannot do that offline, so we use a **proxy**: did the rollout change
anything at all? No patch scores `0`; any patch scores a random number.

Note what that proxy actually rewards — *writing to a file*, not fixing the bug.
A rollout that overwrites `separable.py` with nonsense scores just as well as
one that fixes it. That is what a leaky proxy reward looks like, and it is why
real SWE-bench runs the tests instead.

In [ ]:
def reward_random(patch, rng):
    """Stand-in for a real verifier (unit tests, or a reward model).

    A real verifier applies test_patch plus the candidate patch and runs the
    tests. This one only asks "did the agent change anything?" -- 0 when the
    rollout produced no patch, a random score when it did. Called once per
    rollout, when that episode ends.
    """
    if not patch:
        return 0.0
    return round(float(rng.random()), 3)

---
# 5. Sample a group

GRPO needs several rollouts of the **same** task to compare against each other,
so we run the agent `G = 6` times from the same prompt at `temperature=1.0`.
Same harness, same environment, same prompt — only the model's sampling varies.

In [ ]:
G = 8
rng = np.random.default_rng(0)

group, reward = [], []
for i in range(G-1):
    r     = run_agent(model, tok)              # one rollout: the episode runs to the end
    score = reward_random(r["patch"], rng)     # then, and only then, it gets scored
    group.append(r)
    reward.append(score)
    print(f"rollout {i}: {len(r['calls'])} commands, "
          f"patch {'yes' if r['patch'] else 'no'}, reward {score}")

# Add oracle rollout from 3.3.2, added so the group is never all-zeros.
group.append(expert)
reward.append(reward_random(expert["patch"], rng))

print(f"rollout {G}: {len(expert['calls'])} commands, "
      f"patch yes, reward {reward[-1]}")

reward = np.array(reward)

---
# 6. One GRPO step

$$A_i = \frac{r_i - \mathrm{mean}(r)}{\mathrm{std}(r) + \varepsilon}$$

No critic. **The other rollouts are the baseline** — that is the entire idea.
Loss is a policy gradient over assistant tokens only; observation tokens came
from the environment, so crediting them is meaningless.

**Degenerate groups.** If every rollout scores the same, `std` is 0 — and so is
every numerator, since each reward then equals the mean. Every advantage comes
out 0 and the step is a no-op; `ε` is what makes that a 0 rather than a NaN.

**Why LoRA, when the model is only 0.5B?** Not size — dtype. We loaded the model
`float16` for fast generation, and Adam's second moment underflows there: a
gradient of `1e-4` squares to `1e-8`, below fp16's smallest value, so the
denominator becomes 0 and the first step yields NaN. PEFT keeps the adapter in
fp32, which avoids it. A full finetune would mean reloading in fp32 — about
9.7 GiB instead of 3.2, and half the generation speed.

LoRA also initialises to the identity, so the policy that produced the rollouts
in §5 is exactly the policy we are about to update — the update is on-policy.

In [ ]:
from peft import LoraConfig, get_peft_model

model = get_peft_model(model, LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]))
model.print_trainable_parameters()

In [ ]:
def build_masked(messages, tokenizer, max_len=3072):
    ids, labels, prev = [], [], ""
    for i, m in enumerate(messages):
        cur = tokenizer.apply_chat_template(messages[:i+1], tokenize=False)
        assert cur.startswith(prev), "chat template is not append-only"
        seg = tokenizer(cur[len(prev):], add_special_tokens=False)["input_ids"]
        ids += seg
        labels += seg if m["role"] == "assistant" else [-100]*len(seg)
        prev = cur
    return ids[:max_len], labels[:max_len]


def seq_logprob(messages):
    ids, labs = build_masked(messages, tok)
    t = torch.tensor([ids], device=DEV)
    msk = torch.tensor([[0. if l == -100 else 1. for l in labs]], device=DEV)[:, 1:]
    logits = model(t).logits[:, :-1]
    lp = torch.log_softmax(logits.float(), -1).gather(-1, t[:, 1:].unsqueeze(-1)).squeeze(-1)
    return (lp * msk).sum() / msk.sum().clamp(min=1), msk.sum().item()

In [ ]:
rewards = torch.tensor(reward, dtype=torch.float)
adv = (rewards - rewards.mean()) / (rewards.std(unbiased=False) + 1e-4)

print(f"mean reward {rewards.mean():.3f}   std {rewards.std(unbiased=False):.3f}")
display(pd.DataFrame({
    "rollout":    list(range(len(group))),
    "reward":     rewards.numpy().round(3),
    "advantage":  adv.numpy().round(3),
    "sup_tokens": [int(seq_logprob(g["messages"])[1]) for g in group],
}))

In [ ]:
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-5)

logp_before = [seq_logprob(g["messages"])[0].item() for g in group]

model.train(); opt.zero_grad(set_to_none=True)
loss_total = 0.0
for g, a in zip(group, adv):
    lp, _ = seq_logprob(g["messages"])
    loss = -(a.to(DEV) * lp) / len(group)      # REINFORCE with a group baseline
    loss.backward()
    loss_total += loss.item()

grad_norm = torch.nn.utils.clip_grad_norm_(
    [p for p in model.parameters() if p.requires_grad], 1.0)
opt.step()

logp_after = [seq_logprob(g["messages"])[0].item() for g in group]

print(f"loss {loss_total:+.5f}   grad_norm {grad_norm:.4f}")
display(pd.DataFrame({
    "rollout":     list(range(len(group))),
    "advantage":   [round(a.item(), 3) for a in adv],
    "logp_before": [round(b, 4) for b in logp_before],
    "logp_after":  [round(c, 4) for c in logp_after],
    "delta":       [round(c - b, 5) for b, c in zip(logp_before, logp_after)],
}))

**The mechanism.** Positive advantage pushes a trajectory's log-probability up,
negative pushes it down. No value network, no reward model — just *this
trajectory scored better than its siblings, so make it more likely.*